# SOLID Principles

> Generated from [`08-SOLID-Principles.py`](08-SOLID-Principles.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Five OOP design principles that make code easier to maintain and extend, each shown with a "bad" version and a "good" fix, plus a combined example showing them reinforcing each other.

### Key points
- **S — Single Responsibility**: a class should have only one reason to change. Bad: one `Invoice` class does both math AND printing. Good: split into `Invoice` (math) and `InvoicePrinter` (display) — each has one job.
- **O — Open/Closed**: code should be open for extension but closed for modification. Bad: an `if/elif` chain on `shape["type"]` that needs editing for every new shape. Good: each shape is its own class implementing `area()` — `total_area()` never changes when a new shape is added.
- **L — Liskov Substitution**: a subclass must be usable anywhere its parent is expected, without breaking correctness. Bad: `SquareBad(RectangleBad)` overrides `set_width`/`set_height` to change both dimensions at once, silently breaking code that assumes setting width doesn't touch height.
- **I — Interface Segregation**: don't force a class to implement methods it doesn't need — prefer several small, focused interfaces over one large one. Bad: a fat `WorkerBad` interface forces a `Robot` to implement `eat()`. Good: split into `Workable` and `Eatable`, implement only what applies.
- **D — Dependency Inversion**: depend on abstractions, not concrete implementations. Bad: `NotifierBad` is hard-wired to `EmailSenderBad` inside its own `__init__`. Good: `Notifier` takes any `MessageSender` (an ABC) injected from outside — swapping senders needs no edit to `Notifier`.

### Gotchas (the part worth revising twice)
- **Liskov violations often look "correct" in real-world terms but break code**: a `Square` really *is* a kind of `Rectangle` conceptually, but forcing it to inherit `Rectangle`'s mutable `set_width`/`set_height` interface breaks any code that assumes those two are independent — the fix is usually to not model it as inheritance at all.
- **The combined example (#08) shows SRP + OCP + DIP reinforcing each other**: `NotificationService` has one job (fan out messages — SRP), new channels like `PushSender` are added without touching existing code (OCP), and the dispatcher only ever depends on the `MessageSender` abstraction, never a concrete sender (DIP).
- **SOLID is not dogma — it's a tool, not a checklist**: for a 50-line script or throwaway prototype, applying all five principles is over-engineering — the extra classes/interfaces add indirection with no future change to protect against. Apply it when a codebase is large, long-lived, or has multiple contributors and changing requirements; write the simple direct version first, then introduce structure when duplication/fragility actually shows up as pain.

### When it matters
Reach for SOLID when a codebase will be maintained and extended by more than just you over time — it pays off by making future changes safer and cheaper, at the cost of some upfront indirection.

### One-line recap example
```python
class Notifier:
    def __init__(self, sender: MessageSender):  # depends on abstraction, not EmailSender directly
        self.sender = sender
    def notify(self, message):
        self.sender.send(message)   # any MessageSender works - swap freely (DIP)
```

In [ ]:
from abc import ABC, abstractmethod

## 01. SOLID Principles - Introduction

- Five OOP design principles for code that's easier to maintain and extend.
- S - Single Responsibility   O - Open/Closed        L - Liskov Substitution
- I - Interface Segregation   D - Dependency Inversion

## 02. S - Single Responsibility Principle

- A class should have only one reason to change.

In [ ]:
# Bad: one class does math AND formatting/printing (two reasons to change)
class InvoiceBad:
    def __init__(self, amount):
        self.amount = amount
    def total_with_tax(self):
        return self.amount * 1.1
    def print_invoice(self):
        print(f"Invoice total: {self.total_with_tax()}")

# Good: split into two single-purpose classes
class Invoice:
    def __init__(self, amount):
        self.amount = amount
    def total_with_tax(self):
        return self.amount * 1.1

class InvoicePrinter:
    def print_invoice(self, invoice):
        print(f"Invoice total: {invoice.total_with_tax()}")

InvoicePrinter().print_invoice(Invoice(100))

## 03. O - Open/Closed Principle

- Open for extension, closed for modification: add new behavior without
  editing existing, already-tested code.

In [ ]:
# Bad: adding a new shape means editing this function's if/elif chain
def area_bad(shape):
    if shape["type"] == "circle":
        return 3.14159 * shape["radius"] ** 2
    elif shape["type"] == "square":
        return shape["side"] ** 2
    # every new shape -> another elif here

# Good: new shapes are added as new classes, calculate_area() never changes
class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14159 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

def total_area(shapes):
    return sum(shape.area() for shape in shapes)

print("total area:", total_area([Circle(2), Square(3)]))

## 04. L - Liskov Substitution Principle

- A subclass must be usable anywhere its parent class is expected,
  without breaking correctness.

In [ ]:
# Bad: Square "is-a" Rectangle in real life, but breaks Rectangle's contract
class RectangleBad:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    def set_width(self, width):
        self.width = width
    def set_height(self, height):
        self.height = height

class SquareBad(RectangleBad):
    def set_width(self, width):     # breaks expectation: only width changes
        self.width = self.height = width
    def set_height(self, height):   # same problem for height
        self.width = self.height = height

def resize_test(rect):
    rect.set_width(5)
    rect.set_height(10)
    assert rect.width == 5  # fails for SquareBad - violates substitutability

try:
    resize_test(SquareBad(2, 2))
except AssertionError:
    print("SquareBad violates Liskov substitution (breaks Rectangle's contract)")

resize_test(RectangleBad(2, 2))
print("RectangleBad honors the contract - no error")

## 05. I - Interface Segregation Principle

- Don't force a class to implement methods it doesn't need. Prefer several
  small, focused interfaces over one large one.

In [ ]:
# Bad: one fat interface forces Robot to implement eat() (which makes no sense)
class WorkerBad(ABC):
    @abstractmethod
    def work(self): pass
    @abstractmethod
    def eat(self): pass

# Good: split into focused interfaces, implement only what applies
class Workable(ABC):
    @abstractmethod
    def work(self): pass

class Eatable(ABC):
    @abstractmethod
    def eat(self): pass

class Human(Workable, Eatable):
    def work(self): return "Human working"
    def eat(self): return "Human eating"

class Robot(Workable):  # no eat() forced on it
    def work(self): return "Robot working"

print(Human().work(), "|", Human().eat())
print(Robot().work())

## 06. D - Dependency Inversion Principle

- Depend on abstractions, not concrete implementations. High-level code
  shouldn't be tightly coupled to one specific low-level class.

In [ ]:
# Bad: Notifier is hard-wired to EmailSender - can't swap without editing it
class EmailSenderBad:
    def send(self, message):
        print(f"Emailing: {message}")

class NotifierBad:
    def __init__(self):
        self.sender = EmailSenderBad()  # tight coupling to a concrete class
    def notify(self, message):
        self.sender.send(message)

# Good: Notifier depends on an abstraction; any sender that fits it works
class MessageSender(ABC):
    @abstractmethod
    def send(self, message):
        pass

class EmailSender(MessageSender):
    def send(self, message):
        print(f"Emailing: {message}")

class SMSSender(MessageSender):
    def send(self, message):
        print(f"Texting: {message}")

class Notifier:
    def __init__(self, sender: MessageSender):
        self.sender = sender  # injected - depends on the abstraction only
    def notify(self, message):
        self.sender.send(message)

Notifier(EmailSender()).notify("Hello via email")
Notifier(SMSSender()).notify("Hello via SMS")

## 07. Why SOLID Matters

## 08. Combined Example - SRP + OCP + DIP Together (Plugin-Style Notifications)

- A small, realistic design where three principles reinforce each other:
  SRP (each class does one job), OCP (new channels added without editing
  existing code), DIP (the dispatcher depends on the MessageSender
  abstraction from #06, not on any specific channel).

In [ ]:
class PushSender(MessageSender): # a brand-new channel - existing code untouched (OCP)
    def send(self, message):
        print(f"Push notification: {message}")

class NotificationService: # single responsibility: fan out to registered senders (SRP)
    def __init__(self):
        self.senders = [] # depends only on the MessageSender abstraction (DIP)
    def register(self, sender: MessageSender):
        self.senders.append(sender)
    def notify_all(self, message):
        for sender in self.senders:
            sender.send(message)

print("\n# 08. Combined Example - Plugin-Style Notification System")
service = NotificationService()
service.register(EmailSender())
service.register(SMSSender())
service.register(PushSender()) # added without touching NotificationService at all
service.notify_all("Server restarted")

## 09. When NOT to Apply SOLID

- SOLID pays off when a codebase is large, long-lived, or has multiple
  contributors and changing requirements - the abstractions earn their
  keep by making future changes safer and cheaper.
- For a 50-line script, a one-off data analysis, or a throwaway prototype,
  applying all five principles is over-engineering: extra classes and
  interfaces add indirection with no future change to protect against.
  Write the simple, direct version first; introduce SOLID structure only
  when the pain of NOT having it (duplicated logic, fragile edits) shows up.